# XGBoost baselines: emotions (multi-label) and topic (single-label)

Evaluation protocol, shared with the two neural notebooks through `evaluation.py`:

* 70 / 15 / 15 multi-label-stratified split
* TF-IDF is fit **inside a `Pipeline`**, so vocabulary and IDF come from the
  training split only
* the validation split selects the emotion decision thresholds
* the test split is evaluated exactly once and never used for tuning

`Topic` is a single-label column (12 classes, no comma-separated values), so it
is encoded with `LabelEncoder` and trained as multiclass rather than through
`MultiLabelBinarizer`.


In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
from bnlp import CleanText
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

from evaluation import (
    EMOTIONS,
    SEED,
    best_thresholds,
    emotions_metrics,
    majority_baseline,
    save_metrics,
    set_global_seed,
    split_indices,
    topic_metrics,
)

os.makedirs("results", exist_ok=True)
set_global_seed(SEED)


In [ ]:
df = pd.read_csv("Bangla Emotion Dataset.csv")
df.head()


In [ ]:
df.shape


In [ ]:
df.describe()


In [ ]:
clean_text = CleanText(
    fix_unicode=True,
    unicode_norm=True,
    unicode_norm_form="NFKC",
    remove_url=True,
    remove_email=True,
    remove_emoji=True,
    remove_number=True,
    remove_digits=True,
    remove_punct=True,
    replace_with_url="",
    replace_with_email="",
    replace_with_number="",
    replace_with_digit="",
    replace_with_punct="",
)

df["clean_description"] = df["Data"].apply(clean_text)
df["clean_description"].head(15)


## Targets and split

The split is computed once from the emotion matrix and reused for the topic
target, so both tasks see the same rows in the same split.


In [ ]:
y_emotions = df[EMOTIONS].to_numpy(dtype=int)

topic_encoder = LabelEncoder()
y_topic = topic_encoder.fit_transform(df["Topic"])
print("topic classes:", list(topic_encoder.classes_))

splits = split_indices(y_emotions, seed=SEED)
train_idx, val_idx, test_idx = splits["train"], splits["val"], splits["test"]
print({name: len(idx) for name, idx in splits.items()})

X = df["clean_description"]
X_train, X_val, X_test = X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]

emotions_train, emotions_val, emotions_test = (
    y_emotions[train_idx],
    y_emotions[val_idx],
    y_emotions[test_idx],
)
topic_train, topic_test = y_topic[train_idx], y_topic[test_idx]

baseline = majority_baseline(emotions_test)
print("majority per-label accuracy on test:", round(baseline["per_label_majority_accuracy"], 4))


## Emotions (multi-label)

The vectorizer sits inside the pipeline, so it is fitted on training rows only.
Thresholds are tuned on validation predictions, frozen, and then applied to the
test split.


In [ ]:
emotion_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", MultiOutputClassifier(
        XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=-1)
    )),
])
emotion_pipeline.fit(X_train, emotions_train)


def positive_scores(pipeline, texts):
    """Per-label probability of the positive class, via the pipeline's own vectorizer."""
    matrix = pipeline.named_steps["tfidf"].transform(texts)
    estimators = pipeline.named_steps["clf"].estimators_
    return np.column_stack([est.predict_proba(matrix)[:, 1] for est in estimators])


val_scores = positive_scores(emotion_pipeline, X_val)
thresholds = best_thresholds(emotions_val, val_scores)
print("thresholds from validation:", dict(zip(EMOTIONS, np.round(thresholds, 2))))


In [ ]:
test_scores = positive_scores(emotion_pipeline, X_test)
emotions_test_pred = (test_scores >= thresholds).astype(int)

emotion_results = emotions_metrics(emotions_test, emotions_test_pred)
emotion_results["model"] = "xgboost, tf-idf (train-fitted)"
emotion_results["split"] = "test 15%, unseen during training and threshold selection"
emotion_results["thresholds"] = dict(zip(EMOTIONS, thresholds.tolist()))
emotion_results["majority_baseline"] = baseline

print("macro F1:", round(emotion_results["macro_f1"], 4))
print("micro F1:", round(emotion_results["micro_f1"], 4))
print("subset accuracy:", round(emotion_results["subset_accuracy"], 4))
print("hamming loss:", round(emotion_results["hamming_loss"], 4))
print("majority per-label accuracy:", round(baseline["per_label_majority_accuracy"], 4))
pd.DataFrame(emotion_results["per_label"]).T


## Topic (single-label multiclass)

`Topic` holds exactly one of 12 class names, so this is a 12-way multiclass
problem: `objective="multi:softprob"` with `num_class` taken from the encoder.


In [ ]:
topic_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", XGBClassifier(
        objective="multi:softprob",
        num_class=len(topic_encoder.classes_),
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1,
    )),
])
topic_pipeline.fit(X_train, topic_train)

print("topic accuracy (validation):", round(accuracy_score(topic_encoder.transform(df["Topic"].iloc[val_idx]), topic_pipeline.predict(X_val)), 4))

topic_test_pred = topic_pipeline.predict(X_test)
topic_results = topic_metrics(topic_test, topic_test_pred, list(topic_encoder.classes_))
topic_results["model"] = "xgboost, tf-idf (train-fitted)"
topic_results["split"] = "test 15%, unseen during training"
print("topic accuracy (test):", round(topic_results["accuracy"], 4))
print("topic macro F1 (test):", round(topic_results["macro_f1"], 4))
pd.DataFrame(topic_results["per_class"]).T


## Persist models and metrics

Models and metrics are written to `results/`, so the numbers quoted in the
README are regenerable artifacts rather than notebook output.


In [ ]:
joblib.dump(emotion_pipeline, "results/xgboost_emotions.joblib")
joblib.dump(topic_pipeline, "results/xgboost_topic.joblib")

save_metrics("results/xgboost_emotions.json", emotion_results)
save_metrics("results/xgboost_topic.json", topic_results)
save_metrics("results/xgboost_baseline.json", baseline)
print("saved results/xgboost_*.json and .joblib")


## Predictions on held-out text

Demo inputs are one row from the held-out test split plus a comment that is not
in the dataset at all. Predicting on a training row only demonstrates
memorisation.


In [ ]:
def predict_emotion_and_topic(text):
    cleaned = clean_text(text)
    scores = positive_scores(emotion_pipeline, [cleaned])[0]
    labels = [name for name, score, thr in zip(EMOTIONS, scores, thresholds) if score >= thr]
    topic = topic_encoder.inverse_transform(topic_pipeline.predict([cleaned]))[0]
    return {
        "Emotions": dict(zip(EMOTIONS, np.round(scores, 4))),
        "Predicted labels": labels,
        "Topic": topic,
    }


held_out_index = int(test_idx[0])
print("held-out row:", df.loc[held_out_index, "Data"][:80])
print("gold:", [name for name in EMOTIONS if df.loc[held_out_index, name] == 1], df.loc[held_out_index, "Topic"])
print("prediction:", predict_emotion_and_topic(df.loc[held_out_index, "Data"]))

unseen_text = "আজকের ম্যাচে আমাদের দল দুর্দান্ত খেলেছে, দারুণ জিতেছি"
assert unseen_text not in set(df["Data"])
print("unseen text:", predict_emotion_and_topic(unseen_text))
